# CIMA metabolite/lipid batch GWAS

这个 notebook 做 4 件事：

1. 读取原始 phenotype 表  
2. 把不安全的 phenotype 列名改成适合 PLINK2 的安全列名，并保存映射表  
3. 按 phenotype 全量批量运行 GWAS（无 test mode，支持断点续跑）  
4. 汇总结果，输出 summary 和 manifest  

默认假设：

- 项目根目录：`/data/work/CIMA_multiomics_regulation`
- 基因型前缀：`data/processed/genotype/CIMA_BGEID_metaMatched`
- phenotype：`data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_plink.tsv`
- covariate：`data/processed/meta/CIMA_plink_covariates.tsv`

注意：

- 这里会先生成一个 **safe phenotype file**，后续 GWAS 全用安全列名运行
- `beta` 是在 `--pheno-quantile-normalize` 之后的效应，不是原始代谢物单位
- 默认参数按 `64c / 256g` 机器设置为较激进但通常可用的配置

In [1]:
from pathlib import Path
import os
import re
import json
import shutil
import traceback
import subprocess
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
from IPython.display import display

# =========================
# 项目路径
# =========================
PROJECT_ROOT = Path("/data/work/CIMA_multiomics_regulation")

PFILE_PREFIX = PROJECT_ROOT / "data/processed/genotype/CIMA_BGEID_metaMatched"
PHENO_FILE_RAW = PROJECT_ROOT / "data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_plink.tsv"
PHENO_FILE_SAFE = PROJECT_ROOT / "data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_plink.safe.tsv"
PHENO_NAME_MAP_FILE = PROJECT_ROOT / "data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_name_map.tsv"

COVAR_FILE = PROJECT_ROOT / "data/processed/meta/CIMA_plink_covariates.tsv"

OUT_BASE = PROJECT_ROOT / "data/results/gwas_all"
MANIFEST_DIR = OUT_BASE / "manifest"
STATUS_DONE_DIR = OUT_BASE / "status" / "done"
STATUS_FAILED_DIR = OUT_BASE / "status" / "failed"
LOG_DIR = OUT_BASE / "logs"
PER_PHENO_DIR = OUT_BASE / "per_pheno"
SUMMARY_DIR = OUT_BASE / "summary"

for d in [MANIFEST_DIR, STATUS_DONE_DIR, STATUS_FAILED_DIR, LOG_DIR, PER_PHENO_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# GWAS 参数
# =========================
PLINK2 = shutil.which("plink2") or "plink2"

COVAR_NAMES = ["SEX", "age", "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8", "PC9", "PC10"]

MAF = 0.05
HWE = 1e-6

# 64c / 256g 建议起步
N_WORKERS = 43
PLINK_THREADS = 2
PLINK_MEMORY_MB = 8000

# phenotype 过滤
MIN_NON_MISSING = 50
MIN_UNIQUE_NON_MISSING = 5

# 运行控制
SKIP_DONE = True
OVERWRITE_FAILED = True

TOTAL_THREADS = N_WORKERS * PLINK_THREADS
TOTAL_MEMORY_MB = N_WORKERS * PLINK_MEMORY_MB

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PLINK2:", PLINK2)
print("PHENO_FILE_RAW:", PHENO_FILE_RAW)
print("PHENO_FILE_SAFE:", PHENO_FILE_SAFE)
print("PHENO_NAME_MAP_FILE:", PHENO_NAME_MAP_FILE)
print("COVAR_FILE:", COVAR_FILE)
print("OUT_BASE:", OUT_BASE)
print("N_WORKERS:", N_WORKERS)
print("PLINK_THREADS:", PLINK_THREADS)
print("PLINK_MEMORY_MB:", PLINK_MEMORY_MB)
print("TOTAL_THREADS:", TOTAL_THREADS)
print("TOTAL_MEMORY_MB:", TOTAL_MEMORY_MB)

PROJECT_ROOT: /data/work/CIMA_multiomics_regulation
PLINK2: /data/work/envs/gwas/bin/plink2
PHENO_FILE_RAW: /data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_plink.tsv
PHENO_FILE_SAFE: /data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_plink.safe.tsv
PHENO_NAME_MAP_FILE: /data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_name_map.tsv
COVAR_FILE: /data/work/CIMA_multiomics_regulation/data/processed/meta/CIMA_plink_covariates.tsv
OUT_BASE: /data/work/CIMA_multiomics_regulation/data/results/gwas_all
N_WORKERS: 43
PLINK_THREADS: 2
PLINK_MEMORY_MB: 8000
TOTAL_THREADS: 86
TOTAL_MEMORY_MB: 344000


## Step 1. 基础检查

In [5]:
def check_exists(fp: Path):
    if not fp.exists():
        raise FileNotFoundError(f"文件不存在: {fp}")

for fp in [
    PFILE_PREFIX.with_suffix(".pgen"),
    PFILE_PREFIX.with_suffix(".pvar"),
    PFILE_PREFIX.with_suffix(".psam"),
    PHENO_FILE_RAW,
    COVAR_FILE,
]:
    check_exists(fp)

covar_df = pd.read_csv(COVAR_FILE, sep="\t")
missing_covar = [c for c in COVAR_NAMES if c not in covar_df.columns]
if missing_covar:
    raise ValueError(f"covariate 文件缺少列: {missing_covar}")

print("基础文件检查通过")
print("covar_df shape:", covar_df.shape)
display(covar_df.head(3))

基础文件检查通过
covar_df shape: (443, 14)


,FID,IID,age,SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,E-B21106356138,E-B21106356138,23,1,-0.073847,0.002028,-0.012433,-0.037921,-0.014996,-0.027292,-0.013141,-0.067857,0.064526,0.022914
1,E-B21133356716,E-B21133356716,59,2,-0.049211,0.043807,-0.024668,0.087450,-0.025979,-0.042605,0.012079,-0.027078,-0.015667,0.014470
2,E-B21138997257,E-B21138997257,68,2,0.060666,-0.074132,-0.060308,-0.005670,-0.119269,0.077326,-0.065876,-0.038970,0.021952,0.041801


## Step 2. 生成 safe phenotype 文件

这里把原始 phenotype 列名统一改成只包含字母、数字、下划线的安全名字，避免 `--pheno-name` 被逗号、括号、连字符等字符误解析。

In [6]:
pheno_raw = pd.read_csv(PHENO_FILE_RAW, sep="\t")
print("pheno_raw shape:", pheno_raw.shape)

required_id_cols = ["FID", "IID"]
for c in required_id_cols:
    if c not in pheno_raw.columns:
        raise ValueError(f"表型文件缺少列: {c}")

def make_safe_colname(x: str) -> str:
    x = str(x).strip()
    x = re.sub(r"[^\w]+", "_", x)
    x = re.sub(r"_+", "_", x)
    x = x.strip("_")
    if x == "":
        x = "pheno"
    if re.match(r"^\d", x):
        x = "P_" + x
    return x

old_cols = pheno_raw.columns.tolist()
new_cols = []
used = set()

for c in old_cols:
    if c in ["FID", "IID"]:
        new_c = c
    else:
        base = make_safe_colname(c)
        new_c = base
        i = 1
        while new_c in used:
            i += 1
            new_c = f"{base}__{i}"
    used.add(new_c)
    new_cols.append(new_c)

pheno_safe = pheno_raw.copy()
pheno_safe.columns = new_cols
pheno_safe.to_csv(PHENO_FILE_SAFE, sep="\t", index=False)

name_map = pd.DataFrame({
    "old_name": old_cols,
    "new_name": new_cols
})
name_map.to_csv(PHENO_NAME_MAP_FILE, sep="\t", index=False)

print("safe phenotype file saved:", PHENO_FILE_SAFE)
print("name map saved:", PHENO_NAME_MAP_FILE)
display(name_map.head(20))

pheno_raw shape: (390, 1551)
safe phenotype file saved: /data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_plink.safe.tsv
name map saved: /data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_name_map.tsv


,old_name,new_name
0,FID,FID
1,IID,IID
2,O-Phosphoryl-ethanolamine,O_Phosphoryl_ethanolamine
3,Acesulfame,Acesulfame
4,Acetic acid,Acetic_acid
5,Propanoic acid,Propanoic_acid
6,Glycolic acid,Glycolic_acid
7,But-2-enoic acid,But_2_enoic_acid
8,Valeric acid,Valeric_acid
9,2-Aminoisobutyric acid,P_2_Aminoisobutyric_acid


## Step 3. 构建 phenotype manifest

这里按 **safe 列名** 构建 manifest，并记录原始列名。

In [7]:
name_map = pd.read_csv(PHENO_NAME_MAP_FILE, sep="\t")

candidate_phenos = [c for c in pheno_safe.columns if c not in ["FID", "IID"]]

manifest_rows = []
for pheno in candidate_phenos:
    s_raw = pheno_safe[pheno]

    # 先把纯空白字符串处理成缺失
    s_clean = s_raw.replace(r"^\s*$", pd.NA, regex=True)
    s_num = pd.to_numeric(s_clean, errors="coerce")

    n_non_missing = int(s_num.notna().sum())
    n_missing = int(s_num.isna().sum())
    n_unique_non_missing = int(s_num.dropna().nunique())
    numeric_ok = n_non_missing > 0

    eligible = (
        numeric_ok
        and (n_non_missing >= MIN_NON_MISSING)
        and (n_unique_non_missing >= MIN_UNIQUE_NON_MISSING)
    )

    match = name_map.loc[name_map["new_name"] == pheno, "old_name"]
    original_name = match.iloc[0] if len(match) > 0 else pheno

    manifest_rows.append({
        "phenotype": pheno,
        "phenotype_original": original_name,
        "n_non_missing": n_non_missing,
        "n_missing": n_missing,
        "n_unique_non_missing": n_unique_non_missing,
        "numeric_ok": numeric_ok,
        "eligible": eligible,
    })

manifest = pd.DataFrame(manifest_rows).sort_values(
    ["eligible", "n_non_missing", "phenotype"],
    ascending=[False, False, True]
).reset_index(drop=True)

manifest_file = MANIFEST_DIR / "phenotype_manifest.tsv"
manifest.to_csv(manifest_file, sep="\t", index=False)

phenos_to_run = manifest.loc[manifest["eligible"], "phenotype"].tolist()

print("eligible phenotype count:", int(manifest["eligible"].sum()))
print("ineligible phenotype count:", int((~manifest["eligible"]).sum()))
print("manifest saved:", manifest_file)
display(manifest.head(20))

eligible phenotype count: 1533
ineligible phenotype count: 16
manifest saved: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/manifest/phenotype_manifest.tsv


,phenotype,phenotype_original,n_non_missing,n_missing,n_unique_non_missing,numeric_ok,eligible
0,Acesulfame,Acesulfame,390,0,389,True,True
1,Acetic_acid,Acetic acid,390,0,390,True,True
2,Acetylglycine,Acetylglycine,390,0,390,True,True
3,Adenosine_monophosphate,Adenosine monophosphate,390,0,390,True,True
4,Adipic_acid,Adipic acid,390,0,390,True,True
5,Adrenic_acid,Adrenic acid,390,0,390,True,True
6,Alanylglutamine,Alanylglutamine,390,0,390,True,True
7,Alpha_Linolenic_acid,Alpha-Linolenic acid,390,0,390,True,True
8,Alpha_N_Phenylacetyl_L_glutamine,Alpha-N-Phenylacetyl-L-glutamine,390,0,390,True,True
9,Alpha_ketoisovaleric_acid,Alpha-ketoisovaleric acid,390,0,390,True,True


## Step 4. 定义单 phenotype 运行函数

改动点：

- 全部使用 `safe phenotype file`
- 不再写死 `.glm.linear` 文件名，跑完后自动查找 `.glm.*`
- done 跳过逻辑会同时检查核心输出文件是否存在且非空
- 失败时保留 wrapper log 和 failed 标记

In [8]:
AWK_SCRIPT = r'''BEGIN{FS=OFS="\t"}
NR==1{
    test_col=0
    for(i=1;i<=NF;i++){
        if($i=="TEST"){
            test_col=i
        }
    }
    if(!test_col){
        print "ERROR: TEST column not found" > "/dev/stderr"
        exit 1
    }
    print
    next
}
$test_col=="ADD"{print}
'''

def find_glm_output(pheno_dir: Path, pheno_name: str) -> Path:
    cands = sorted(pheno_dir.glob(f"{pheno_name}.{pheno_name}.glm.*"))
    cands = [x for x in cands if x.is_file() and not str(x).endswith(".add")]
    if len(cands) == 0:
        raise FileNotFoundError(f"未找到 glm 输出文件: {pheno_name}")
    if len(cands) > 1:
        # 优先 linear，其次 logistic，其余取第一个
        linear = [x for x in cands if x.name.endswith(".glm.linear")]
        logistic = [x for x in cands if x.name.endswith(".glm.logistic")]
        if linear:
            return linear[0]
        if logistic:
            return logistic[0]
    return cands[0]

def run_one_pheno(pheno_name: str):
    pheno_dir = PER_PHENO_DIR / pheno_name
    pheno_dir.mkdir(parents=True, exist_ok=True)

    out_prefix = pheno_dir / pheno_name
    snplist_file = pheno_dir / f"{pheno_name}.snplist"
    log_file = pheno_dir / f"{pheno_name}.log"
    wrapper_log = LOG_DIR / f"{pheno_name}.wrapper.log"

    done_flag = STATUS_DONE_DIR / f"{pheno_name}.done.json"
    failed_flag = STATUS_FAILED_DIR / f"{pheno_name}.failed.txt"

    glm_file = None
    add_file = None

    if done_flag.exists():
        try:
            old_done = json.loads(done_flag.read_text(encoding="utf-8"))
            old_glm = Path(old_done.get("glm_file", "")) if old_done.get("glm_file") else None
            old_add = Path(old_done.get("add_file", "")) if old_done.get("add_file") else None
            old_snplist = Path(old_done.get("snplist_file", "")) if old_done.get("snplist_file") else None

            if (
                SKIP_DONE
                and old_glm is not None and old_glm.exists() and old_glm.stat().st_size > 0
                and old_add is not None and old_add.exists() and old_add.stat().st_size > 0
                and old_snplist is not None and old_snplist.exists() and old_snplist.stat().st_size > 0
            ):
                return {
                    "phenotype": pheno_name,
                    "status": "skipped_done",
                    "glm_file": str(old_glm),
                    "add_file": str(old_add),
                    "snplist_file": str(old_snplist),
                    "log_file": str(log_file),
                    "wrapper_log": str(wrapper_log),
                    "n_add_rows": old_done.get("n_add_rows"),
                    "n_snps": old_done.get("n_snps"),
                    "start_time": old_done.get("start_time"),
                    "finished_at": old_done.get("finished_at"),
                    "elapsed_seconds": old_done.get("elapsed_seconds"),
                    "error": None,
                }
        except Exception:
            pass

    if OVERWRITE_FAILED and failed_flag.exists():
        failed_flag.unlink()

    cmd = [
        PLINK2,
        "--pfile", str(PFILE_PREFIX),
        "--pheno", str(PHENO_FILE_SAFE),
        "--pheno-name", pheno_name,
        "--covar", str(COVAR_FILE),
        "--covar-name", ",".join(COVAR_NAMES),
        "--maf", str(MAF),
        "--hwe", str(HWE),
        "--pheno-quantile-normalize",
        "--glm", "hide-covar",
        "--write-snplist",
        "--threads", str(PLINK_THREADS),
        "--memory", str(PLINK_MEMORY_MB),
        "--out", str(out_prefix),
    ]

    start_dt = datetime.now()
    start_time = start_dt.isoformat(timespec="seconds")

    try:
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=False
        )

        with open(wrapper_log, "w", encoding="utf-8") as f:
            f.write("CMD:\n")
            f.write(" ".join(cmd) + "\n\n")
            f.write("START:\n")
            f.write(start_time + "\n\n")
            f.write("RETURN_CODE:\n")
            f.write(str(proc.returncode) + "\n\n")
            f.write("STDOUT:\n")
            f.write(proc.stdout or "")
            f.write("\n\nSTDERR:\n")
            f.write(proc.stderr or "")

        if proc.returncode != 0:
            err_msg = f"plink2 returncode={proc.returncode}"
            with open(failed_flag, "w", encoding="utf-8") as f:
                f.write(err_msg + "\n")
                f.write("\nWRAPPER_LOG:\n")
                f.write(str(wrapper_log) + "\n")
            return {
                "phenotype": pheno_name,
                "status": "failed",
                "glm_file": None,
                "add_file": None,
                "snplist_file": str(snplist_file),
                "log_file": str(log_file),
                "wrapper_log": str(wrapper_log),
                "n_add_rows": None,
                "n_snps": None,
                "start_time": start_time,
                "finished_at": datetime.now().isoformat(timespec="seconds"),
                "elapsed_seconds": (datetime.now() - start_dt).total_seconds(),
                "error": err_msg,
            }

        if not snplist_file.exists():
            raise FileNotFoundError(f"plink 输出缺失: {snplist_file}")

        glm_file = find_glm_output(pheno_dir, pheno_name)
        add_file = Path(str(glm_file) + ".add")

        with open(add_file, "w", encoding="utf-8") as fout:
            awk_proc = subprocess.run(
                ["awk", AWK_SCRIPT, str(glm_file)],
                stdout=fout,
                stderr=subprocess.PIPE,
                text=True,
                check=False
            )
        if awk_proc.returncode != 0:
            raise RuntimeError(f"awk 提取 ADD 失败: {awk_proc.stderr}")

        n_add_rows = max(sum(1 for _ in open(add_file, "r", encoding="utf-8")) - 1, 0)
        n_snps = sum(1 for _ in open(snplist_file, "r", encoding="utf-8"))

        end_dt = datetime.now()
        done_info = {
            "phenotype": pheno_name,
            "status": "done",
            "glm_file": str(glm_file),
            "add_file": str(add_file),
            "snplist_file": str(snplist_file),
            "log_file": str(log_file),
            "wrapper_log": str(wrapper_log),
            "n_add_rows": n_add_rows,
            "n_snps": n_snps,
            "start_time": start_time,
            "finished_at": end_dt.isoformat(timespec="seconds"),
            "elapsed_seconds": (end_dt - start_dt).total_seconds(),
            "error": None,
        }

        with open(done_flag, "w", encoding="utf-8") as f:
            json.dump(done_info, f, ensure_ascii=False, indent=2)

        return done_info

    except Exception as e:
        err_text = f"{type(e).__name__}: {e}\n{traceback.format_exc()}"
        with open(failed_flag, "w", encoding="utf-8") as f:
            f.write(err_text)

        return {
            "phenotype": pheno_name,
            "status": "failed",
            "glm_file": str(glm_file) if glm_file else None,
            "add_file": str(add_file) if add_file else None,
            "snplist_file": str(snplist_file),
            "log_file": str(log_file),
            "wrapper_log": str(wrapper_log),
            "n_add_rows": None,
            "n_snps": None,
            "start_time": start_time,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "elapsed_seconds": (datetime.now() - start_dt).total_seconds(),
            "error": str(e),
        }

print("run_one_pheno() 已定义")

run_one_pheno() 已定义


## Step 5. 全量并行运行 batch GWAS

这里不再使用 test mode，直接对所有 eligible phenotype 运行。

In [9]:
results = []

summary_file = SUMMARY_DIR / "gwas_batch_summary.tsv"

# 先读已有 summary，支持断点续跑
if summary_file.exists():
    old_summary = pd.read_csv(summary_file, sep="\t")
else:
    old_summary = pd.DataFrame()

# 已完成的不再提交
if SKIP_DONE and not old_summary.empty and "status" in old_summary.columns:
    done_set = set(old_summary.loc[old_summary["status"].isin(["done", "skipped_done"]), "phenotype"].astype(str))
else:
    done_set = set()

phenos_pending = [ph for ph in phenos_to_run if ph not in done_set]

print("\n=== start batch GWAS ===")
print("eligible phenotype count:", len(phenos_to_run))
print("already done:", len(done_set))
print("pending:", len(phenos_pending))
print("summary_file:", summary_file)

def save_summary(old_df, new_results, out_file):
    new_df = pd.DataFrame(new_results)

    if old_df is not None and len(old_df) > 0:
        merged = pd.concat([old_df, new_df], ignore_index=True)
    else:
        merged = new_df.copy()

    if len(merged) > 0:
        merged = merged.drop_duplicates(subset=["phenotype"], keep="last")
        merged = merged.sort_values(["status", "phenotype"]).reset_index(drop=True)

    tmp_file = out_file.with_suffix(".tmp.tsv")
    merged.to_csv(tmp_file, sep="\t", index=False)
    tmp_file.replace(out_file)

if len(phenos_pending) > 0:
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        future_map = {ex.submit(run_one_pheno, ph): ph for ph in phenos_pending}

        for i, fut in enumerate(as_completed(future_map), start=1):
            ph = future_map[fut]
            try:
                res = fut.result()
            except Exception as e:
                res = {
                    "phenotype": ph,
                    "status": "failed",
                    "glm_file": None,
                    "add_file": None,
                    "snplist_file": None,
                    "log_file": None,
                    "wrapper_log": None,
                    "n_add_rows": None,
                    "n_snps": None,
                    "start_time": None,
                    "finished_at": datetime.now().isoformat(timespec="seconds"),
                    "elapsed_seconds": None,
                    "error": f"executor exception: {e}",
                }

            results.append(res)

            # 每完成一个就立刻写 summary
            save_summary(old_summary, results, summary_file)

            print(
                f"[{i}/{len(phenos_pending)}] "
                f"{res['phenotype']} -> {res['status']} "
                f"(n_add_rows={res['n_add_rows']}, n_snps={res['n_snps']}, elapsed={res['elapsed_seconds']})"
            )
else:
    print("nothing to run")

print("\n=== batch finished ===")

# 最后再读一次完整 summary
summary_df = pd.read_csv(summary_file, sep="\t") if summary_file.exists() else pd.DataFrame()
display(summary_df.tail())


=== start batch GWAS ===
eligible phenotype count: 1533
already done: 0
pending: 1533
summary_file: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/summary/gwas_batch_summary.tsv


[1/1533] Acesulfame -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=145.166958)
[2/1533] Alpha_ketoisovaleric_acid -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=129.152582)
[3/1533] Aspartame -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=117.619917)
[4/1533] Adipic_acid -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=138.978177)
[5/1533] Alanylglutamine -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=144.952124)
[6/1533] Adrenic_acid -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=144.498165)
[7/1533] Alpha_Linolenic_acid -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=145.31061)
[8/1533] Adenosine_monophosphate -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=144.008789)
[9/1533] Arabinonic_acid -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=106.429672)
[10/1533] Alpha_N_Phenylacetyl_L_glutamine -> skipped_done (n_add_rows=7405576, n_snps=6656848, elapsed=99.315584)

,phenotype,status,glm_file,add_file,snplist_file,log_file,wrapper_log,n_add_rows,n_snps,start_time,finished_at,elapsed_seconds,error
1528,myo_Inositol,skipped_done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:28:30,2026-04-17T12:31:30,180.268397,NaN
1529,p_Hydroxymandelic_acid,skipped_done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:28:31,2026-04-17T12:31:27,175.960493,NaN
1530,p_Hydroxyphenylacetic_acid,skipped_done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:28:31,2026-04-17T12:31:30,179.506394,NaN
1531,s_5_Adenosyl_L_homocysteine,skipped_done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:28:34,2026-04-17T12:31:33,179.070614,NaN
1532,trans_Vaccenic_acid,skipped_done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:28:37,2026-04-17T12:31:14,157.087728,NaN


## Step 6. 汇总结果并回写原始 phenotype 名

In [10]:
summary = pd.DataFrame(results)

if len(summary) == 0:
    raise RuntimeError("summary 为空，说明没有任何 phenotype 被提交运行")

summary = summary.merge(
    name_map.rename(columns={"new_name": "phenotype", "old_name": "phenotype_original"}),
    on="phenotype",
    how="left"
)

summary = summary.sort_values(
    ["status", "phenotype"],
    ascending=[True, True]
).reset_index(drop=True)

summary_file = SUMMARY_DIR / "gwas_run_summary.tsv"
summary.to_csv(summary_file, sep="\t", index=False)

print("summary saved:", summary_file)
print(summary["status"].value_counts(dropna=False))
display(summary.head(20))

summary saved: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/summary/gwas_run_summary.tsv
status
done            516
failed          513
skipped_done    504
Name: count, dtype: int64


,phenotype,status,glm_file,add_file,snplist_file,log_file,wrapper_log,n_add_rows,n_snps,start_time,finished_at,elapsed_seconds,error,phenotype_original
0,Cer_d18_1_22_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:56:31,2026-04-17T12:58:39,128.412035,None,Cer d18:1/22:0
1,Cer_d18_1_24_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:49:32,2026-04-17T12:51:44,131.608883,None,Cer d18:1/24:0
2,Cer_d18_1_24_1,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:04:25,2026-04-17T13:07:16,170.436648,None,Cer d18:1/24:1
3,DAG_16_0_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:58:36,2026-04-17T13:01:33,176.777417,None,DAG 16:0-18:3
4,DAG_16_1_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:02:07,2026-04-17T13:04:31,143.552243,None,DAG 16:1-18:3
5,DAG_16_1_18_4,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:02:27,2026-04-17T13:04:39,131.927172,None,DAG 16:1-18:4
6,DAG_18_0_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:55:36,2026-04-17T12:58:21,165.147652,None,DAG 18:0-18:3
7,DAG_18_1_18_4,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:04:18,2026-04-17T13:06:41,142.968390,None,DAG 18:1-18:4
8,DAG_18_1_20_2,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:55:31,2026-04-17T12:57:18,106.838177,None,DAG 18:1-20:2
9,DAG_18_1_20_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:43:24,2026-04-17T12:46:07,163.135135,None,DAG 18:1-20:3


## Step 7. 失败任务快速排查

如果有失败 phenotype，先看 wrapper log。

In [11]:
failed = summary[summary["status"] == "failed"].copy()
print("failed count:", failed.shape[0])

if failed.shape[0] > 0:
    display(failed[[
        "phenotype", "phenotype_original", "error", "wrapper_log"
    ]].head(20))

    first_log = failed["wrapper_log"].dropna().iloc[0] if failed["wrapper_log"].dropna().shape[0] > 0 else None
    if first_log and Path(first_log).exists():
        print("\n===== first failed wrapper log preview =====\n")
        with open(first_log, "r", encoding="utf-8") as f:
            print(f.read()[:5000])
else:
    print("没有 failed phenotype")

failed count: 513


,phenotype,phenotype_original,error,wrapper_log
516,PC_16_0_16_0,PC 16:0-16:0,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
517,PC_16_0_16_1,PC 16:0-16:1,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
518,PC_16_0_18_0,PC 16:0-18:0,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
519,PC_16_0_18_1,PC 16:0-18:1,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
520,PC_16_0_18_2,PC 16:0-18:2,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
521,PC_16_0_18_3,PC 16:0-18:3,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
522,PC_16_0_18_4,PC 16:0-18:4,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
523,PC_16_0_20_0,PC 16:0-20:0,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
524,PC_16_0_20_1,PC 16:0-20:1,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...
525,PC_16_0_20_2,PC 16:0-20:2,plink2 returncode=6,/data/work/CIMA_multiomics_regulation/data/res...



===== first failed wrapper log preview =====

CMD:
/data/work/envs/gwas/bin/plink2 --pfile /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched --pheno /data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_plink.safe.tsv --pheno-name PC_16_0_16_0 --covar /data/work/CIMA_multiomics_regulation/data/processed/meta/CIMA_plink_covariates.tsv --covar-name SEX,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 --maf 0.05 --hwe 1e-06 --pheno-quantile-normalize --glm hide-covar --write-snplist --threads 2 --memory 8000 --out /data/work/CIMA_multiomics_regulation/data/results/gwas_all/per_pheno/PC_16_0_16_0/PC_16_0_16_0

START:
2026-04-17T13:01:43

RETURN_CODE:
6

STDOUT:
PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /data/work/CIMA_multiomics_regulation/data/results/gwas_all/per_pheno/PC_16_0_16_0/PC_16_

## 结果目录说明

- `data/results/gwas_batch/manifest/phenotype_manifest.tsv`  
  phenotype manifest

- `data/results/gwas_batch/per_pheno/<phenotype>/`  
  每个 phenotype 的 plink 输出

- `data/results/gwas_batch/logs/<phenotype>.wrapper.log`  
  包装层日志，优先看这个

- `data/results/gwas_batch/status/done/*.done.json`  
  已完成状态

- `data/results/gwas_batch/status/failed/*.failed.txt`  
  失败状态

- `data/results/gwas_batch/summary/gwas_run_summary.tsv`  
  总汇总表

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/data/work/CIMA_multiomics_regulation")
OUT_BASE = PROJECT_ROOT / "data/results/gwas_all"

MANIFEST_FILE = OUT_BASE / "manifest" / "phenotype_manifest.tsv"
BATCH_SUMMARY_FILE = OUT_BASE / "summary" / "gwas_batch_summary.tsv"
RUN_SUMMARY_FILE = OUT_BASE / "summary" / "gwas_run_summary.tsv"

FIG_DIR = OUT_BASE / "figures_weekly_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("MANIFEST_FILE:", MANIFEST_FILE)
print("BATCH_SUMMARY_FILE:", BATCH_SUMMARY_FILE)
print("RUN_SUMMARY_FILE:", RUN_SUMMARY_FILE)
print("FIG_DIR:", FIG_DIR)

if not MANIFEST_FILE.exists():
    raise FileNotFoundError(f"manifest 不存在: {MANIFEST_FILE}")

manifest = pd.read_csv(MANIFEST_FILE, sep="\t")
print("manifest shape:", manifest.shape)
display(manifest.head())

batch_summary = None
if BATCH_SUMMARY_FILE.exists():
    batch_summary = pd.read_csv(BATCH_SUMMARY_FILE, sep="\t")
    print("batch_summary shape:", batch_summary.shape)
    display(batch_summary.head())

run_summary = None
if RUN_SUMMARY_FILE.exists():
    run_summary = pd.read_csv(RUN_SUMMARY_FILE, sep="\t")
    print("run_summary shape:", run_summary.shape)
    display(run_summary.head())

MANIFEST_FILE: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/manifest/phenotype_manifest.tsv
BATCH_SUMMARY_FILE: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/summary/gwas_batch_summary.tsv
RUN_SUMMARY_FILE: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/summary/gwas_run_summary.tsv
FIG_DIR: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/figures_weekly_report
manifest shape: (1549, 7)


,phenotype,phenotype_original,n_non_missing,n_missing,n_unique_non_missing,numeric_ok,eligible
0,Acesulfame,Acesulfame,390,0,389,True,True
1,Acetic_acid,Acetic acid,390,0,390,True,True
2,Acetylglycine,Acetylglycine,390,0,390,True,True
3,Adenosine_monophosphate,Adenosine monophosphate,390,0,390,True,True
4,Adipic_acid,Adipic acid,390,0,390,True,True


batch_summary shape: (1533, 13)


,phenotype,status,glm_file,add_file,snplist_file,log_file,wrapper_log,n_add_rows,n_snps,start_time,finished_at,elapsed_seconds,error
0,Cer_d18_1_22_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:56:31,2026-04-17T12:58:39,128.412035,NaN
1,Cer_d18_1_24_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:49:32,2026-04-17T12:51:44,131.608883,NaN
2,Cer_d18_1_24_1,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:04:25,2026-04-17T13:07:16,170.436648,NaN
3,DAG_16_0_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:58:36,2026-04-17T13:01:33,176.777417,NaN
4,DAG_16_1_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:02:07,2026-04-17T13:04:31,143.552243,NaN


run_summary shape: (1533, 14)


,phenotype,status,glm_file,add_file,snplist_file,log_file,wrapper_log,n_add_rows,n_snps,start_time,finished_at,elapsed_seconds,error,phenotype_original
0,Cer_d18_1_22_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:56:31,2026-04-17T12:58:39,128.412035,NaN,Cer d18:1/22:0
1,Cer_d18_1_24_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:49:32,2026-04-17T12:51:44,131.608883,NaN,Cer d18:1/24:0
2,Cer_d18_1_24_1,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:04:25,2026-04-17T13:07:16,170.436648,NaN,Cer d18:1/24:1
3,DAG_16_0_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:58:36,2026-04-17T13:01:33,176.777417,NaN,DAG 16:0-18:3
4,DAG_16_1_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:02:07,2026-04-17T13:04:31,143.552243,NaN,DAG 16:1-18:3
